In [ ]:
# ============================================================
# CELL 1 — MASTER FEATURE SCHEMA (Single Source of Truth)
# Run this cell first before anything else
# ============================================================
import numpy as np
import pandas as pd

LOGON_FEATURES = [
    'total_events',
    'off_hours_ratio',
    'unique_systems_accessed',
    'failed_login_ratio',
    'active_orphan_sessions'
]

HTTP_FEATURES = [
    'url_visits',
    'bytes_uploaded',
    'malicious_domain_hits',
    'unique_external_domains',
    'after_hours_browsing'
]

EMAIL_FEATURES = [
    'emails_sent',
    'external_recipients',
    'after_hours_emails',
    'attachments_sent',
    'large_attachment_count'
]

MASTER_FEATURE_SCHEMA = LOGON_FEATURES + HTTP_FEATURES + EMAIL_FEATURES


def align_to_master_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensures DataFrame always conforms to master feature schema.
    Missing columns (http, email) filled with NaN until those
    sources are pulled. Enforces consistent column ordering.
    """
    for col in MASTER_FEATURE_SCHEMA:
        if col not in df.columns:
            df[col] = np.nan
    return df[MASTER_FEATURE_SCHEMA]


In [ ]:
# ============================================================
# CELL 2 — EXTRACTION PIPELINE
# ============================================================
import duckdb
import logging
from pathlib import Path
from datetime import datetime

# Configure standard logging (Industry Best Practice)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def extract_behavioral_features(user_input_path: str, dev_mode: bool = False) -> str:
    """
    Securely extracts a time-bucketed behavioral feature matrix from raw logon logs.
    """
    logging.info("Initiating Feature Extraction Pipeline...")

    # ============================================================
    # 1. SECURITY SANDBOX & PATH VALIDATION
    # ============================================================
    if dev_mode:
        logging.warning("Running in DEV_MODE. Strict security sandbox is DISABLED.")
        target_file: Path = Path(user_input_path).resolve()

        if not target_file.exists():
            logging.error(f"DEV ERROR: Target file not found at {target_file}")
            raise FileNotFoundError(f"Cannot find file at {target_file}")
    else:
        # PROD MODE: Enforce the Sandbox
        safe_directory: Path = Path("/path/to/your/secure/datasets/").resolve()
        target_file: Path = (safe_directory / user_input_path).resolve()

        if not target_file.is_relative_to(safe_directory):
            logging.error(f"SECURITY ALERT: Directory Traversal Attempt Detected: {user_input_path}")
            raise ValueError("Access Denied: Invalid file path.")

    if target_file.suffix != '.csv':
        logging.error(f"SECURITY ALERT: Invalid file type attempted: {target_file.suffix}")
        raise ValueError("Access Denied: Only .csv files are permitted.")

    target_file_str = str(target_file)
    if "'" in target_file_str:
        raise ValueError("File path contains illegal character. Single quotes are not permitted.")

    # ============================================================
    # 2. OUTPUT PATH SETUP
    # ============================================================
    output_dir: Path = Path("/path/to/your/secure/features/").resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file: Path = output_dir / f"logon_features_{timestamp}.parquet"
    output_file_str = str(output_file)

    if "'" in output_file_str:
        raise ValueError("Output path contains illegal character. Single quotes are not permitted.")

    # ============================================================
    # 3. SQL ARCHITECTURE (UPDATED)
    # ============================================================
    logging.info(f"Ingesting raw logs from: {target_file}")
    logging.info("Applying Time-Bucketed CTE — extracting daily behavioral features...")

    query: str = f"""
    COPY (
        -- ── STEP 1: Deduplication & Sessionization ──────────────────────
        WITH SessionizedLogs AS (
            SELECT
                user,
                pc,          
                activity,    
                TIME_BUCKET(
                    INTERVAL '5 minutes',
                    TRY_CAST(date AS TIMESTAMP)
                ) AS session_time
            FROM read_csv_auto('{target_file_str}')
            WHERE user != 'SYSTEM'
                -- [FIX 2] Added 'Logoff' to the filter to track session closure
                AND (activity IN ('Logon', 'Logoff') OR activity ILIKE '%fail%')
            GROUP BY user, pc, activity, session_time
        )

        -- ── STEP 2: Per-User DAILY Feature Aggregation ───────────────────
        SELECT
            user,
            CAST(session_time AS DATE) AS activity_date, -- [FIX 1] Temporal Grouping

            COUNT(*) AS total_events,

            SUM(
                CASE
                    WHEN DAYOFWEEK(session_time) IN (0, 6) THEN 1   -- Weekend
                    WHEN EXTRACT(hour FROM session_time) < 6
                      OR EXTRACT(hour FROM session_time) >= 18 THEN 1 -- Night hours
                    ELSE 0
                END
            ) * 1.0 / NULLIF(COUNT(*), 0) AS off_hours_ratio,

            COUNT(DISTINCT pc) AS unique_systems_accessed,

            SUM(
                CASE WHEN activity ILIKE '%fail%' THEN 1 ELSE 0 END
            ) * 1.0 / NULLIF(COUNT(*), 0) AS failed_login_ratio,

            -- [FIX 2] Orphan Session Indicator (Logons minus Logoffs)
            -- A high positive number means they are leaving sessions open 
            -- (Lateral movement or scripted activity)
            SUM(CASE WHEN activity = 'Logon' THEN 1 ELSE 0 END) -
            SUM(CASE WHEN activity = 'Logoff' THEN 1 ELSE 0 END) AS active_orphan_sessions

        FROM SessionizedLogs
        GROUP BY user, CAST(session_time AS DATE) -- [FIX 1] Group by User AND Date

    ) TO '{output_file_str}' (FORMAT PARQUET);
    """

    # ============================================================
    # 4. EXECUTION BLOCK
    # ============================================================
    try:
        duckdb.sql(query)
    except Exception as e:
        logging.error(f"[extract_behavioral_features] DuckDB execution failed. Reason: {e}")
        raise

    # ============================================================
    # 5. POST-EXECUTION VALIDATION + SCHEMA ALIGNMENT
    # ============================================================
    df_check = pd.read_parquet(output_file)

    if len(df_check) == 0:
        raise RuntimeError("Pipeline produced empty output. Check WHERE filters.")

    # [FIX 1 Update] The index is now a MultiIndex: User AND Date.
    # This is required for time-series anomaly detection.
    df_aligned = align_to_master_schema(df_check.set_index(['user', 'activity_date']))

    df_aligned.to_parquet(output_file, index=True)

    logging.info(
        f"Pipeline Complete. | "
        f"Rows extracted: {len(df_aligned)} | "
        f"Schema: {list(df_aligned.columns)} | "
        f"Output: {output_file}"
    )

    return str(output_file)

In [ ]:
result = extract_behavioral_features('CERT_dataset/r4.2/logon.csv', dev_mode=True)

In [ ]:
import pandas as pd
import numpy as np
import logging
import joblib
from datetime import datetime
from sklearn.impute import SimpleImputer
from scipy.stats import median_abs_deviation
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest

# Configure strict production logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class InsiderThreatDetector:
    """
    Dual model ML pipeline for Insider Threat Detection.
    Features: NaN Auditing, Immutable Features, Provenance Tracking, and Dynamic Scoring.
    """
    def __init__(self, mad_threshold: float = 3.5, missing_data_tolerance: float = 0.2):
        # Parameterized Thresholds
        self.mad_threshold = mad_threshold
        self.missing_data_tolerance = missing_data_tolerance
        
        # contamination='auto' allows dynamic scoring via decision_function()
        self.iso_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
            ('scaler', RobustScaler()), 
            ('detector', IsolationForest(contamination='auto', random_state=42, n_estimators=100))
        ])
        
        self.baseline_stats = {}
        self.is_fitted = False
        self.model_version = None

    @classmethod
    def load(cls, model_path: str) -> "InsiderThreatDetector":
        """
        Deserializes a saved InsiderThreatDetector from disk.
        Validates structural integrity before returning the object.
    
        Usage:
            detector = InsiderThreatDetector.load("iso_pipeline_v20240101_120000.pkl")
        """
        logging.info(f"[load] Deserializing model from: {model_path}")
    
        try:
            obj = joblib.load(model_path)
        except FileNotFoundError:
            raise FileNotFoundError(
                f"[load] Model file not found at: '{model_path}'. "
                f"Verify the artifact path or rerun fit_baseline()."
        )
        except Exception as e:
            raise RuntimeError(
            f"[load] Failed to deserialize model. "
            f"File may be corrupted or incompatible. Reason: {e}"
        )

        # ── Integrity checks on the loaded object ─────────────────────────────
        if not isinstance(obj, cls):
            raise TypeError(
                f"[load] Loaded object is type '{type(obj).__name__}', "
                f"expected 'InsiderThreatDetector'. Wrong file loaded."
            )
        if not obj.is_fitted:
            raise RuntimeError(
                f"[load] Loaded model has is_fitted=False. "
                f"This model was serialized before training completed."
            )
        if not obj.baseline_stats:
            raise RuntimeError(
            f"[load] Loaded model has empty baseline_stats. "
            f"MAD scoring will be non-functional. Retrain and re-serialize."
        )

        logging.info(
            f"[load] Model loaded successfully | "
            f"version={obj.model_version} | "
            f"features={len(obj.baseline_stats)} | "
            f"iso_threshold={getattr(obj, 'iso_threshold', 'NOT FOUND — retrain recommended')}"
        )
    
        return obj


    def _validate_input(self, df: pd.DataFrame, context: str) -> None:
        """
        Gate-check for all incoming DataFrames.
        Called before any model training or inference begins.
        Raises ValueError with actionable messages on any violation.
        """

        # ── CHECK 1: Empty DataFrame ──────────────────────────────────────────
        if len(df) == 0:
            raise ValueError(
                f"[{context}] Empty DataFrame received. "
                f"Check if the upstream ETL pipeline produced output."
            )

        # ── CHECK 2: 'user' column existence ─────────────────────────────────
        if 'user' not in df.columns:
            raise ValueError(
                f"[{context}] Required column 'user' is missing. "
                f"Columns present: {list(df.columns)}. "
                f"Check if ETL pipeline renamed the identifier column."
            )

        # ── CHECK 3: Duplicate users ──────────────────────────────────────────
        # Must be checked BEFORE set_index() — after indexing, duplicates are harder to surface
        if df['user'].duplicated().any():
            dupes = df['user'][df['user'].duplicated()].tolist()
            raise ValueError(
                f"[{context}] Duplicate users detected: {dupes[:5]}{'...' if len(dupes) > 5 else ''}. "
                f"Resolve deduplication in the upstream join logic."
            )

        # ── CHECK 4: No numeric columns present ──────────────────────────────
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) == 0:
            raise ValueError(
                f"[{context}] No numeric feature columns found. "
                f"All columns are non-numeric: {list(df.columns)}. "
                f"Check feature engineering pipeline output."
            )

        # ── CHECK 5: Feature drift (inference only) ───────────────────────────
        # Only runs after the model is fitted — compares live columns to trained columns
        if self.is_fitted and context == "predict_live_traffic":
            trained_features = set(self.baseline_stats.keys())
            live_features = set(df.select_dtypes(include=[np.number]).columns.tolist())
            missing_features = trained_features - live_features
            new_features = live_features - trained_features

            if missing_features:
                raise ValueError(
                    f"[{context}] FEATURE DRIFT DETECTED. "
                    f"Columns trained on but missing from live data: {missing_features}. "
                    f"Model cannot score reliably. Retrain or fix ETL pipeline."
                )
            if new_features:
                # New features are a warning, not a hard stop — model ignores unknown columns
                logging.warning(
                    f"[{context}] New unseen columns in live data: {new_features}. "
                    f"These will be ignored by the model. Consider retraining."
                )

        logging.info(f"[{context}] Input validation passed. Rows: {len(df)} | Numeric features: {len(numeric_cols)}")


    def fit_baseline(self, historical_parquet_path: str):
        """Trains the model on known-clean historical data and exports the serialized model."""
        logging.info(f"TRAINING PHASE: Loading historical baseline from {historical_parquet_path}")
        
        # ── STEP 1: Safe file loading ─────────────────────────────────────────
        try:
            df_raw = pd.read_parquet(historical_parquet_path)
        except FileNotFoundError:
            raise FileNotFoundError(
                f"[fit_baseline] Parquet file not found at: '{historical_parquet_path}'. "
                f"Verify the path and check if the ETL job completed successfully."
            )
        except Exception as e:
                raise RuntimeError(f"[fit_baseline] Failed to load parquet file. Reason: {e}")

        # ── STEP 2: Validate BEFORE touching the data ─────────────────────────
        self._validate_input(df_raw, context="fit_baseline")

            
           
        df_features = df_raw.set_index('user')

        # 1. Learn the MAD Baseline
        numeric_cols = df_features.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            col_median = df_features[col].median(skipna=True)
            col_mad = median_abs_deviation(df_features[col].dropna(), scale='normal')
            self.baseline_stats[col] = {'median': col_median, 'mad': col_mad}

        logging.info(
            f"[fit_baseline] MAD baseline computed | "
            f"features={len(self.baseline_stats)} | "
            f"zero_mad_cols={[c for c,v in self.baseline_stats.items() if v['mad'] == 0]}"
        )


        # 2.  The pipeline now handles imputation internally
        self.iso_pipeline.fit(df_features)
        self.is_fitted = True
        # ── Auto-derive ISO threshold from training distribution ──────────────
        train_scores = self.iso_pipeline.decision_function(df_features)
        # Threshold = mean - 1 std: flags the statistically anomalous tail
        self.iso_threshold = float(np.mean(train_scores) - np.std(train_scores))

        logging.info(
            f"[fit_baseline] ISO threshold auto-derived | "
            f"mean_score={np.mean(train_scores):.4f} | "
            f"std={np.std(train_scores):.4f} | "
            f"threshold={self.iso_threshold:.4f}"
        )
        
        # Model Serialization & Provenance
        self.model_version = datetime.now().strftime("%Y%m%d_%H%M%S")
        export_path = f"iso_pipeline_v{self.model_version}.pkl"
        joblib.dump(self, export_path)
        logging.info(
            f"[fit_baseline] Model serialized | version={self.model_version} | "
            f"path={export_path} | features={list(self.baseline_stats.keys())}"
        )

        return export_path
        

    def predict_live_traffic(self, live_parquet_path: str, iso_decision_threshold: float = None) -> pd.DataFrame:
        """
        Evaluates new traffic. 
        Returns ONLY the df_scores DataFrame, leaving the raw features untouched.
        """
        if not self.is_fitted:
            raise RuntimeError(
                "Model must be fit() before predict() is called."
                 "Run fit_baseline() first."  
            )
        # ── Resolve threshold: caller override → auto-derived → hard fallback ─
        if iso_decision_threshold is None:
            if hasattr(self, 'iso_threshold'):
                iso_decision_threshold = self.iso_threshold
                logging.info(f"[predict_live_traffic] Using auto-derived threshold: {iso_decision_threshold:.4f}")
            else:
                iso_decision_threshold = -0.1  # last-resort fallback, logged clearly
                logging.warning(
                    "[predict_live_traffic] iso_threshold not found on model. "
                    "Falling back to default -0.1. Retrain model to fix this."
                )

        try:
            df_raw = pd.read_parquet(live_parquet_path)
        except FileNotFoundError:
            raise FileNotFoundError(
                f"[predict_live_traffic] Parquet file not found at: '{live_parquet_path}'. "
                f"Verify the path and check if the live ingestion job completed."
            )
        except Exception as e:
            raise RuntimeError(f"[predict_live_traffic] Failed to load parquet file. Reason: {e}")

        # ── STEP 2: Validate BEFORE touching the data ─────────────────────────
        # Feature drift check is ACTIVE here because is_fitted=True
        self._validate_input(df_raw, context="predict_live_traffic")

        # ── Everything below this line is guaranteed safe ─────────────────────
        df_features = df_raw.set_index('user')
        logging.info(f"INFERENCE PHASE: Evaluating live traffic from {live_parquet_path}")
        
        # Strict Separation of Concerns (Create a totally separate DF for outputs)
        df_scores = pd.DataFrame(index=df_features.index)

        # =====================================================================
        # The NaN Audit (Blind Spot Detection)
        # =====================================================================
        # Calculate percentage of missing values per user
        missing_pct = df_features.isnull().mean(axis=1)
        df_scores['data_quality_risk'] = np.where(missing_pct > self.missing_data_tolerance, 1, 0)
        
        if df_scores['data_quality_risk'].sum() > 0:
            logging.warning(f"BLIND SPOT ALERT: {df_scores['data_quality_risk'].sum()} users flagged for suspicious data loss.")


        # =====================================================================
        # MODEL 1: Apply the LOCKED Historical MAD Baseline
        # =====================================================================
        # MAD scoring requires no NaNs — imputed separately from the sklearn pipeline
        # This does NOT affect ISO Forest, which handles imputation internally

        mad_flags_matrix = pd.DataFrame(index=df_features.index)
        df_for_mad = df_features.fillna(0)


        for col in self.baseline_stats.keys():
            if col in df_for_mad.columns:
                hist_median = self.baseline_stats[col]['median']
                hist_mad = self.baseline_stats[col]['mad']
                live_values = df_for_mad[col]

                if hist_mad == 0:
                    z_scores = np.where(live_values > hist_median, 10.0, 0.0)
                else:
                    z_scores = np.abs(0.6745 * (live_values - hist_median) / hist_mad)
                
                # Using the parameterized threshold (self.mad_threshold)
                mad_flags_matrix[f'{col}_flag'] = np.where(z_scores > self.mad_threshold, 1, 0)

        df_scores['mad_score_count'] = mad_flags_matrix.sum(axis=1)
        df_scores['mad_critical_flag'] = np.where(df_scores['mad_score_count'] >= 2, 1, 0)

        logging.info(
            f"[predict_live_traffic] MAD scoring complete | "
            f"critical_flags={df_scores['mad_critical_flag'].sum()} | "
            f"avg_flag_count={df_scores['mad_score_count'].mean():.2f}"
        )


        # =====================================================================
        # MODEL 2: Dynamic Isolation Forest Inference
        # =====================================================================
        df_scores['iso_forest_raw_score'] = self.iso_pipeline.decision_function(df_features)
        
        # Apply an adjustable threshold instead of trusting hard binary predictions
        df_scores['iso_forest_flag'] = np.where(df_scores['iso_forest_raw_score'] < iso_decision_threshold, 1, 0)

        logging.info(
            f"[predict_live_traffic] ISO Forest scoring complete | "
            f"score_range=[{df_scores['iso_forest_raw_score'].min():.4f}, "
            f"{df_scores['iso_forest_raw_score'].max():.4f}] | "
            f"threshold={iso_decision_threshold:.4f} | "
            f"flagged={df_scores['iso_forest_flag'].sum()}"
        )


        # =====================================================================
        # DATA FUSION: Final Threat Assessment
        # =====================================================================
        # We also ensure we don't trust the model if the data quality is compromised
        # =====================================================================
        # DATA FUSION: Final Threat Assessment (Security-Hardened)
        # =====================================================================

        # CASE 1: Both models agree AND data is clean → high-confidence confirmed threat
        df_scores['confirmed_threat'] = np.where(
            (df_scores['mad_critical_flag'] == 1) &
            (df_scores['iso_forest_flag'] == 1) &
            (df_scores['data_quality_risk'] == 0),
            1, 0
        )

        # CASE 2: Both models agree BUT data is suspicious → escalate, do NOT suppress
        # An attacker corrupting their telemetry + triggering both models = highest risk
        df_scores['high_risk_review'] = np.where(
            (df_scores['mad_critical_flag'] == 1) &
            (df_scores['iso_forest_flag'] == 1) &
            (df_scores['data_quality_risk'] == 1),
            1, 0
        )

        # CASE 3: Data quality alone is suspicious → flag for analyst review
        # Even with no model signal, unexplained data loss is an IOC (Indicator of Compromise)
        df_scores['data_loss_ioc'] = np.where(
            (df_scores['data_quality_risk'] == 1) &
            (df_scores['confirmed_threat'] == 0) &
            (df_scores['high_risk_review'] == 0),
            1, 0
        )

        logging.info(
            f"[predict_live_traffic] Threat summary | "
            f"confirmed={df_scores['confirmed_threat'].sum()} | "
            f"high_risk_review={df_scores['high_risk_review'].sum()} | "
            f"data_loss_ioc={df_scores['data_loss_ioc'].sum()}"
            )

        
        # We ONLY return the notebook of scores. The raw evidence remains unmutated.
        return df_scores 